In [0]:
%sql
CREATE TABLE IF NOT EXISTS migration.silver.customers_scd2
(
    customer_id BIGINT,
    customer_name STRING,
    email STRING,
    city STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    -- SCD Type 2 columns
    effective_from TIMESTAMP,
    effective_to TIMESTAMP,
    is_current BOOLEAN,

    -- Pipeline metadata
    _source_batch_id STRING,
    _source_run_id STRING,
    _source_ingestion_timestamp TIMESTAMP,
    _processed_timestamp TIMESTAMP
)
USING DELTA;

# Initial SCD Load

In [0]:
# %sql
# INSERT INTO migration.silver.customers_scd2
# SELECT
#     customer_id,
#     customer_name,
#     email,
#     city,
#     created_at,
#     updated_at,

#     -- SCD Type 2
#     updated_at AS effective_from,
#     CAST(NULL AS TIMESTAMP) AS effective_to,
#     TRUE AS is_current,

#     -- Pipeline metadata
#     _source_batch_id,
#     _source_run_id,
#     _source_ingestion_timestamp,
#     current_timestamp() AS _processed_timestamp

# FROM migration.silver.customers;

# Preparing source

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW scd2_changes AS

SELECT
    s.*
FROM migration.silver.customers s
INNER JOIN migration.silver.customers_scd2 t
    ON s.customer_id = t.customer_id
   AND t.is_current = true
WHERE s.updated_at > t.updated_at;

In [0]:
%sql

SELECT
    customer_id,
    customer_name,
    city,
    updated_at
FROM scd2_changes
ORDER BY customer_id;

# Close the old version

In [0]:
%sql

MERGE INTO migration.silver.customers_scd2 AS target

USING scd2_changes AS source

ON target.customer_id = source.customer_id
AND target.is_current = true

WHEN MATCHED
AND source.updated_at > target.updated_at

THEN UPDATE SET
    target.effective_to = source.updated_at,
    target.is_current = false;

In [0]:
%sql

SELECT
    customer_id,
    customer_name,
    city,
    effective_from,
    effective_to,
    is_current
FROM migration.silver.customers_scd2
WHERE customer_id = 7
ORDER BY effective_from;

# Insert the new version

In [0]:
%sql

INSERT INTO migration.silver.customers_scd2
SELECT
    customer_id,
    customer_name,
    email,
    city,
    created_at,
    updated_at,

    -- New version starts at the source change time
    updated_at AS effective_from,

    -- Current version has no end date yet
    CAST(NULL AS TIMESTAMP) AS effective_to,

    -- This is the active version
    TRUE AS is_current,

    -- Pipeline metadata
    _source_batch_id,
    _source_run_id,
    _source_ingestion_timestamp,
    current_timestamp() AS _processed_timestamp

FROM scd2_changes;

In [0]:
%sql

SELECT
    customer_id,
    customer_name,
    city,
    effective_from,
    effective_to,
    is_current
FROM migration.silver.customers_scd2
WHERE customer_id = 7
ORDER BY effective_from;

# ------------------error---------------

In [0]:
%sql

UPDATE migration.silver.customers_scd2
SET
    effective_to = (
        SELECT MIN(effective_from)
        FROM migration.silver.customers_scd2
        WHERE customer_id = 1
          AND city = 'Pune'
    ),
    is_current = false
WHERE customer_id = 1
  AND city = 'Nashik'
  AND is_current = true;

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT_IF(is_current = true) AS current_records,
    COUNT_IF(is_current = false) AS historical_records
FROM migration.silver.customers_scd2;